In [1]:
import pandas as pd
import re
import string
import nltk
import emoji

from tqdm import tqdm
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [2]:
resources = [
    "punkt",
    "punkt_tab",
    "stopwords"
]

for resource in resources:
    nltk.download(resource)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Erland\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Erland\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Erland\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
df = pd.read_csv("../datasets/X_review_GP/interim/cleaned_X_reviews.csv")
df = df[["Ulasan", "Rating"]]
df.columns = ["text", "rating"]
df.head()

,text,rating
0,"ya,bagus",4
1,mohon bantuannya saya tidak bisa mengubah nama...,2
2,tolong lah X (sebelumnya Twiter) akun saya tid...,3
3,update selalu 🫰🏻,5
4,"sangat menyebalkan, hampir setiap membuka apk ...",1


## 1. Label Mapping

In [4]:
def label_sentiment(rating):
    if rating >= 4:
        return "positive"
    elif rating == 3:
        return "neutral"
    else:
        return "negative"

In [5]:
df["label"] = df["rating"].apply(label_sentiment)
df[["rating", "label"]].head()

,rating,label
0,4,positive
1,2,negative
2,3,neutral
3,5,positive
4,1,negative


## 2. Stopword & Stemmer

In [6]:
# Stopword Indonesia
stop_words = set(stopwords.words('indonesian'))

# Stemmer Indonesia
factory = StemmerFactory()
stemmer = factory.create_stemmer()

## 3. Cleaning Text

In [7]:
def case_folding(text):
    text = text.lower()
    return text

def normalize_repeated_characters(text):
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

def remove_url(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', text)

def remove_mention_hashtag(text):
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    return text

def remove_emoji(text):
    return emoji.replace_emoji(text, replace='')

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def remove_punctuation(text):
    return re.sub(
        f"[{re.escape(string.punctuation)}]",
        " ",
        text
    )

def remove_whitespace(text):
    return text.strip()

def remove_multiple_spaces(text):
    return re.sub(r'\s+', ' ', text)

## 4. Tokenisasi

In [8]:
def tokenize(text):
    return word_tokenize(text)

## 5. Stopword Removal & Stemming

In [9]:
# Stopword Removal
def remove_stopwords(tokens):
    return [
        word for word in tokens
        if word not in stop_words
    ]

# Stemming
def stemming(text):
    return stemmer.stem(text)

## 6. Full Preprocessing Function

In [10]:
def preprocess_text(text):
    text = case_folding(text)
    text = normalize_repeated_characters(text)
    text = remove_url(text)
    text = remove_mention_hashtag(text)
    text = remove_emoji(text)
    text = remove_numbers(text)
    text = remove_punctuation(text)
    text = remove_whitespace(text)
    text = remove_multiple_spaces(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    text = ' '.join(tokens)
    text = stemming(text)
    return text

## Testing

In [11]:
sample = "Gw suka bangettt aplikasi ini 😭🔥!!!"

print(preprocess_text(sample))

gw suka bangett aplikasi


In [12]:
tqdm.pandas()

df["clean_text"] = df["text"].progress_apply(preprocess_text)
# df["clean_text"] = df["text"].apply(preprocess_text)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 75434/75434 [1:23:33<00:00, 15.05it/s]


In [13]:
df[["text", "clean_text"]].head()

,text,clean_text
0,"ya,bagus",ya bagus
1,mohon bantuannya saya tidak bisa mengubah nama...,mohon bantu ubah nama guna
2,tolong lah X (sebelumnya Twiter) akun saya tid...,tolong x twiter akun upload langgar tdk koment...
3,update selalu 🫰🏻,update
4,"sangat menyebalkan, hampir setiap membuka apk ...",sebal buka apk x pinta verifikasi email


## 7. Clean Final Dataset

In [14]:
df = df[df["clean_text"].str.strip() != ""]

In [15]:
# Cek duplikat setelah preprocessing
print(f"\nJumlah data duplikat: {df.duplicated(subset=["clean_text"]).sum()}")

# Membersihkan data duplikat
df = df.drop_duplicates(subset=["clean_text"])

# Jumlah data duplikat setelah dibersihkan
print(f"\nJumlah data duplikat setelah dibersihkan: {df.duplicated(subset=["clean_text"]).sum()}")


Jumlah data duplikat: 9908

Jumlah data duplikat setelah dibersihkan: 0


In [16]:
# Jumlah kolom dan baris setelah dibersihkan
df.shape

(64341, 4)

# 8. Final Check Dataset

In [17]:
# Struktur kolom
print(f"\nKolom pada dataset: {df.columns}")

# Tipe data
print("\nTipe data tiap kolom")
print("=" * 30)
print(df.info())

# Missing values
print("\n\nMissing Values")
print("=" * 30)
print(df.isnull().sum())

# Empty text
print(f"\n\nJumlah Empty Text: {(df["clean_text"].str.strip() == "").sum()}")

# Duplicate pada Raw text & clean text
print(f"\n\nJumlah duplikat data pada raw text (Sebelum preprocessing): {df.duplicated(subset=["text"]).sum()}")
print(f"Jumlah duplikat data pada clean text (Sesudah preprocessing): {df.duplicated(subset=["clean_text"]).sum()}")

# Label Validation
print("\n\nLabel Validation")
print("=" * 30)
print(df["label"].unique())

# Distribusi Label
print("\n\nDistribusi Label")
print("=" * 30)
print(df["label"].value_counts())


Kolom pada dataset: Index(['text', 'rating', 'label', 'clean_text'], dtype='str')

Tipe data tiap kolom
<class 'pandas.DataFrame'>
Index: 64341 entries, 0 to 75432
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   text        64341 non-null  str  
 1   rating      64341 non-null  int64
 2   label       64341 non-null  str  
 3   clean_text  64341 non-null  str  
dtypes: int64(1), str(3)
memory usage: 2.5 MB
None


Missing Values
text          0
rating        0
label         0
clean_text    0
dtype: int64


Jumlah Empty Text: 0


Jumlah duplikat data pada raw text (Sebelum preprocessing): 0
Jumlah duplikat data pada clean text (Sesudah preprocessing): 0


Label Validation
<StringArray>
['positive', 'negative', 'neutral']
Length: 3, dtype: str


Distribusi Label
label
negative    41994
positive    18544
neutral      3803
Name: count, dtype: int64


In [24]:
df[["text", "clean_text", "label"]].sample(10)

,text,clean_text,label
30831,apk gajelas masukin nomor telepon ga bisa bisa,apk gajelas masukin nomor telepon ga,negative
5070,akun gua di tangguhkan Mulu padahal ga aneh aneh,akun gua tangguh mulu ga aneh aneh,negative
4146,kok susah dibuka sekarang,susah buka,positive
60681,ada aGaRp ra,agarp ra,positive
35189,Brengsek sulit login,brengsek sulit login,negative
9440,error parah,error parah,negative
19124,Saya bisa mendapat kan isu isu yg lagi viral,isu isu yg viral,positive
26608,Aplikasi ya bagus si Tapi kenapa setiap buat a...,aplikasi ya bagus si akun akun kunci,positive
14151,aplikasi selalu merekomendasikan hal hal yang ...,aplikasi rekomendasi senonoh,negative
1204,gw baru download lagi trs gabisa di buka????? ...,gw download trs gabisa buka kekk ih tolol,negative


In [26]:
all_words = ' '.join(df["clean_text"])

words = all_words.split()

print(words[:500])

['ya', 'bagus', 'mohon', 'bantu', 'ubah', 'nama', 'guna', 'tolong', 'x', 'twiter', 'akun', 'upload', 'langgar', 'tdk', 'komentar', 'yg', 'langgar', 'kena', 'suspend', 'tangguh', 'update', 'sebal', 'buka', 'apk', 'x', 'pinta', 'verifikasi', 'email', 'mantap', 'x', 'login', 'apk', 'muncul', 'error', 'login', 'tolong', 'baik', 'akun', 'akun', 'login', 'apknya', 'batiba', 'sign', 'out', 'susah', 'banget', 'alias', 'ga', 'sign', 'in', 'bikin', 'emosi', 'sebal', 'bagus', 'oke', 'tingkat', 'kwalitas', 'nya', 'aplikasi', 'aneh', 'baperan', 'upload', 'apa', 'akun', 'kena', 'suspend', 'tangguh', 'coba', 'banding', 'tolak', 'tolong', 'lha', 'x', 'baik', 'jariangan', 'bagus', 'batas', 'jelek', 'payah', 'mantaapp', 'x', 'bantu', 'berita', 'kini', 'dunia', 'parahh', 'akun', 'nya', 'tangguh', 'good', 'apls', 'seruu', 'aplikasi', 'nya', 'buka', 'gajelas', 'atur', 'nya', 'gatau', 'salah', 'nya', 'tiba', 'tangguh', 'permanen', 'apk', 'nya', 'jelek', 'login', 'sush', 'ngebug', 'gajelas', 'login', 'tolong

In [20]:
# Cek Panjang text
df["text_length"] = df["clean_text"].apply(len)
df["text_length"].describe()

count    64341.000000
mean        41.572994
std         40.836424
min          1.000000
25%         16.000000
50%         29.000000
75%         52.000000
max        488.000000
Name: text_length, dtype: float64

In [21]:
# Text sangat pendek
df[df["text_length"] < 3]

,text,rating,label,clean_text,text_length
42,ok,5,positive,ok,2
139,Ga jelas,5,positive,ga,2
3354,oj,5,positive,oj,2
3888,up,5,positive,up,2
4238,ya begitulah,5,positive,ya,2
...,...,...,...,...,...
70705,. Bbb,5,positive,bb,2
72639,",Oa 😁",4,positive,oa,2
72743,AT,4,positive,at,2
73018,Ik,5,positive,ik,2


## 9. Save Final Dataset

In [27]:
df.to_csv(
    "../datasets/X_review_GP/processed/final_clean_dataset.csv",
    index=False
)